In [214]:
#!python -m spacy download en_core_web_sm

In [215]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [216]:
# # modeli za ustrezno obdelavo stavkov, besed, ločil....
# nltk.download('punkt')     # stavki, besede
# nltk.download('wordnet') #lemmatizacija
# nltk.download('averaged_perceptron_tagger') #POS tagganje
# nltk.download('omw-1.4') 
# nltk.download('punkt_tab')
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

In [217]:
# # tokenization and lemmatization 
# lemmatizer= WordNetLemmatizer()

In [218]:
# # pokupčkamo besede s podobnim korenom, pomenom skupaj
# # run, runs, running -> run
# def get_wordnet_pos(treebank_tag):
#     if treebank_tag.startswith('J'):
#         return wordnet.ADJ
#     elif treebank_tag.startswith('V'):
#         return wordnet.VERB
#     elif treebank_tag.startswith('RB'):
#         return wordnet.ADV
#     elif treebank_tag.startswith('N'):
#         return wordnet.NOUN
#     else:
#         return wordnet.NOUN

In [219]:
import spacy

nlp = spacy.load('en_core_web_sm')

In [220]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti osebe, kraji, jeziki, narodi...
    odstrani_pos = ['PROPN', 'PRON', 'VERB', 'ADV']
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    #mnozica prepoznanih entitet
    for token in doc:
        # odstranimo i
        if token.pos_ in odstrani_pos:
            continue
        if token.ent_type_ in odstrani_entitete:
            continue
            
        # spacy ima oznake NOUN, ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            # samo crke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [221]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major', 'character', 'cover', 'magazine',
    'self', 'series', 'volume', 'masterpiece', 'translation', 'film', 'tale', 'course',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 'isbn', 'mother', 'sister', 'father',
    'little', 'room', 'place', 'end', 'first', 'second',
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [222]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\03_ang_opisi\*.txt')[:150]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words= all_stopwords, #custom_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.7)

In [223]:
X = vectorizer.fit_transform(filepaths) 

c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [224]:
words = vectorizer.get_feature_names_out()
freq = np.asarray(X.sum(axis=0)).flatten()

top_words = [words[i] for i in freq.argsort()[-50:]]
print(top_words)

['evil', 'passion', 'plan', 'age', 'prison', 'black', 'different', 'town', 'master', 'trouble', 'perfect', 'career', 'husband', 'journey', 'killer', 'hand', 'lady', 'warrior', 'body', 'business', 'murder', 'truth', 'cat', 'high', 'dead', 'country', 'mysterious', 'dream', 'house', 'adventure', 'work', 'century', 'key', 'marriage', 'night', 'wife', 'beautiful', 'school', 'home', 'heart', 'power', 'city', 'death', 'child', 'dark', 'friend', 'war', 'history', 'family', 'love']


In [225]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2807 stored elements and shape (150, 551)>
  Coords	Values
  (0, 166)	1
  (0, 128)	2
  (0, 405)	1
  (0, 334)	1
  (0, 350)	1
  (0, 414)	1
  (0, 369)	1
  (0, 509)	1
  (0, 354)	1
  (0, 378)	1
  (0, 213)	1
  (0, 190)	2
  (0, 466)	1
  (0, 105)	1
  (0, 464)	1
  (0, 121)	1
  (0, 200)	1
  (0, 1)	1
  (0, 487)	1
  (0, 120)	1
  (0, 367)	1
  (0, 292)	1
  (1, 128)	1
  (1, 200)	1
  (1, 181)	1
  :	:
  (147, 534)	3
  (147, 207)	1
  (147, 119)	1
  (147, 87)	1
  (147, 67)	3
  (147, 511)	1
  (147, 242)	1
  (147, 335)	1
  (147, 54)	1
  (147, 276)	1
  (147, 330)	1
  (147, 444)	1
  (148, 334)	1
  (148, 350)	1
  (148, 375)	1
  (148, 73)	1
  (149, 354)	1
  (149, 105)	1
  (149, 224)	1
  (149, 326)	1
  (149, 421)	1
  (149, 115)	1
  (149, 550)	2
  (149, 57)	1
  (149, 177)	2
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 2]]
   account  action  actual  addition  adolescence  adventure  

In [226]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [227]:
# test 

W, H, errors = nmf(X, 4)
print(errors)
#print(W)
#print(H)

[np.float64(64.43286594712954), np.float64(63.93317692596921), np.float64(63.58420410077793), np.float64(63.269511471637166), np.float64(63.02672094859751), np.float64(62.84416001377562), np.float64(62.69969066479107), np.float64(62.58150335360172), np.float64(62.48345303844078), np.float64(62.39477579891986), np.float64(62.30517018004608), np.float64(62.209542298542694), np.float64(62.11022674952734), np.float64(62.015145526118076), np.float64(61.93168372751232), np.float64(61.86390112614674), np.float64(61.80977839277887), np.float64(61.76451835369174), np.float64(61.72886588284877), np.float64(61.70302908681911), np.float64(61.68327030683497), np.float64(61.6664276539136), np.float64(61.65396745220822), np.float64(61.643590495212365), np.float64(61.63416207741244), np.float64(61.62554975518059), np.float64(61.618355715892925), np.float64(61.61251036469024), np.float64(61.60739968924843), np.float64(61.60309301261828), np.float64(61.59964225113507), np.float64(61.59703374390648), np.

In [230]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-10:]]
    print(f"Tema {topic_idx +1}: {' '.join(top_words)}")

Tema 1: perfect friend beautiful wife love house marriage school child family
Tema 2: land power heart art bloody dark war empire city love
Tema 3: belief violence savage realm bone work religious fundamentalist violent faith
Tema 4: century death resource epic revolution war historical country camp history


In [229]:
import os

# iz W dobim, kateri temi pripada katera knjiga, dobimo indeks najvišje vrednosti v vsaki vrstici
dominant_topics = np.argmax(W, axis=1)+1

filenames = [os.path.basename(f) for f in filepaths]

#tabela
df_results = pd.DataFrame({
    'Knjiga': filenames,
    'Tema': dominant_topics
})
print(df_results.head(10))
print(df_results[df_results['Tema'] == 4])
print(df_results[df_results['Tema'] == 5]) # za specif. teme
print(df_results[df_results['Tema'] == 2])


         Knjiga  Tema
0    opis_1.txt     1
1   opis_10.txt     1
2  opis_100.txt     2
3  opis_101.txt     3
4  opis_102.txt     1
5  opis_103.txt     1
6  opis_104.txt     3
7  opis_105.txt     4
8  opis_106.txt     4
9  opis_107.txt     4
           Knjiga  Tema
7    opis_105.txt     4
8    opis_106.txt     4
9    opis_107.txt     4
10   opis_108.txt     4
14   opis_111.txt     4
18   opis_115.txt     4
22   opis_119.txt     4
29   opis_125.txt     4
33   opis_129.txt     4
38   opis_133.txt     4
40   opis_135.txt     4
46   opis_140.txt     4
60   opis_153.txt     4
63   opis_156.txt     4
68   opis_160.txt     4
70   opis_162.txt     4
90   opis_180.txt     4
91   opis_181.txt     4
95   opis_185.txt     4
114   opis_30.txt     4
146    opis_6.txt     4
Empty DataFrame
Columns: [Knjiga, Tema]
Index: []
           Knjiga  Tema
2    opis_100.txt     2
11   opis_109.txt     2
13   opis_110.txt     2
15   opis_112.txt     2
26   opis_122.txt     2
30   opis_126.txt     2
50   opis_14